# Questão 8 — Sistema de Recomendação

### Contexto

Implementar uma vitrine de **'Quem comprou isso, também levou...'** usando similaridade de comportamento de compra entre clientes.

Produto de referência: GPS Garmin Vortex Maré Drift

Será feito uma filtragem Colaborativa baseada em itens (Item-Based Collaborative Filtering)

---

### Biblioteca de machine learning

In [9]:
import subprocess
subprocess.run(['pip', 'install', 'scikit-learn'], capture_output=True)

CompletedProcess(args=['pip', 'install', 'scikit-learn'], returncode=0, stdout=b'Requirement already satisfied: scikit-learn in /home/renan/lighthouse_project/.venv/lib/python3.12/site-packages (1.8.0)\nRequirement already satisfied: numpy>=1.24.1 in /home/renan/lighthouse_project/.venv/lib/python3.12/site-packages (from scikit-learn) (2.4.3)\nRequirement already satisfied: scipy>=1.10.0 in /home/renan/lighthouse_project/.venv/lib/python3.12/site-packages (from scikit-learn) (1.17.1)\nRequirement already satisfied: joblib>=1.3.0 in /home/renan/lighthouse_project/.venv/lib/python3.12/site-packages (from scikit-learn) (1.5.3)\nRequirement already satisfied: threadpoolctl>=3.2.0 in /home/renan/lighthouse_project/.venv/lib/python3.12/site-packages (from scikit-learn) (3.6.0)\n', stderr=b'')

### Carregando os dados

In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df_vendas   = pd.read_csv('../data/raw/vendas_2023_2024.csv')
df_produtos = pd.read_csv('../data/raw/produtos_raw.csv')
df_produtos = df_produtos.drop_duplicates(subset='code', keep='first')

print(f'Vendas:   {len(df_vendas)} registros')
print(f'Produtos: {len(df_produtos)} registros')


Vendas:   9895 registros
Produtos: 150 registros


### Identificando o produto de referência

In [11]:
produto_alvo = df_produtos[df_produtos['name'] == 'GPS Garmin Vortex Maré Drift']
print(produto_alvo[['code', 'name']])

TARGET_ID = produto_alvo['code'].iloc[0]
print(f'\nID do produto alvo: {TARGET_ID}')


    code                          name
26    27  GPS Garmin Vortex Maré Drift

ID do produto alvo: 27


### Matriz Usuário x Produto

A matriz é binária: cada célula vale `1` se o cliente comprou ao menos uma vez aquele produto, e `0` caso contrário. A quantidade comprada é ignorada — só importa presença ou ausência.

```
         prod_1  prod_2  prod_3  ...
cliente1    1       0       1
cliente2    0       1       1
cliente3    1       1       0
```

In [12]:
# Agrupando por cliente e produto
df_interacao = df_vendas.groupby(['id_client', 'id_product'])['qtd'].sum().reset_index()
df_interacao['comprou'] = 1  # binariza: qualquer qtd vira 1

matriz = df_interacao.pivot_table(
    index='id_client',
    columns='id_product',
    values='comprou',
    fill_value=0
)

print(f'Shape da matriz: {matriz.shape}')
print(f'Clientes: {matriz.shape[0]} | Produtos: {matriz.shape[1]}')
matriz.head()


Shape da matriz: (49, 150)
Clientes: 49 | Produtos: 150


id_product,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
id_client,,,,,,,,,,,,,,,,,,,,,
1,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
2,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,...,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
4,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0
5,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0


### Calculando a Similaridade de Cosseno entre Produtos

Medindo o angulo entre dois vetores. Quanto mais próximo de `1`, mais similares os produtos — ou seja, os mesmos clientes tendem a comprar os dois.

Para calcular produto x produto, transfere-se a matriz (produtos viram linhas, clientes viram colunas). Assim cada produto é representado por um vetor indicando quais clientes o compraram.

```
similaridade = cos(theta) = (A . B) / (||A|| * ||B||)
```

In [13]:
# .T transpõe a matriz: produtos viram linhas
sim_matrix = cosine_similarity(matriz.T)

# Transforma em DataFrame para facilitar a leitura
sim_df = pd.DataFrame(
    sim_matrix,
    index=matriz.columns,
    columns=matriz.columns
)

print(f'Matriz de similaridade: {sim_df.shape}')
print('Similaridade do produto alvo com os primeiros 5 produtos:')
print(sim_df[TARGET_ID].head())


Matriz de similaridade: (150, 150)
Similaridade do produto alvo com os primeiros 5 produtos:
id_product
1    0.850000
2    0.748331
3    0.764217
4    0.784873
5    0.694879
Name: 27, dtype: float64


### Gerando o ranking dos 5 produtos mais similares

In [14]:
# Pega a coluna do produto alvo, remove ele mesmo, ordena decrescente
ranking = (
    sim_df[TARGET_ID]
    .drop(TARGET_ID)              # remove o proprio produto
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
ranking.columns = ['id_produto', 'similaridade']

# Adiciona o nome do produto
ranking = ranking.merge(
    df_produtos[['code', 'name']],
    left_on='id_produto',
    right_on='code'
).drop(columns='code')

ranking.index += 1  # ranking comeca em 1
ranking[['id_produto', 'name', 'similaridade']]


,id_produto,name,similaridade
1,94,Motor de Popa Volvo Magnum 276HP,0.869626
2,11,GPS Furuno Swift Leviathan Poseidon,0.868037
3,35,Radar Furuno Swift,0.853913
4,1,Transponder AIS Maré Magnum,0.850000
5,115,Cabo de Nylon Delta Force Magnum Leviathan,0.850000


### Exibir a vitrine de recomendação

In [15]:
print('Produto de referencia:')
print(f'  GPS Garmin Vortex Maré Drift (id={TARGET_ID})')
print()
print('Quem comprou isso, também levou:')
print('-' * 55)
for i, row in ranking.iterrows():
    print(f'  {i}. {row["name"]}')
    print(f'     Similaridade: {row["similaridade"]:.4f}')


Produto de referencia:
  GPS Garmin Vortex Maré Drift (id=27)

Quem comprou isso, também levou:
-------------------------------------------------------
  1. Motor de Popa Volvo Magnum 276HP
     Similaridade: 0.8696
  2. GPS Furuno Swift Leviathan Poseidon
     Similaridade: 0.8680
  3. Radar Furuno Swift
     Similaridade: 0.8539
  4. Transponder AIS Maré Magnum
     Similaridade: 0.8500
  5. Cabo de Nylon Delta Force Magnum Leviathan
     Similaridade: 0.8500


---
### A Similaridade de Cosseno funciona dessa forma

Cada produto é representado por um vetor binário com 49 posições (uma por cliente):
- Posição `i = 1` se o cliente `i` comprou o produto
- Posição `i = 0` se nao comprou

A similaridade de cosseno mede o quanto dois vetores apontam na mesma direção. Dois produtos com similaridade `1.0` foram comprados exatamente pelos mesmos clientes. Similaridade `0.0` significa que nenhum cliente comprou os dois.

### Questão 8.1
Mesmo código do notebook, porém em arquivo .py

In [16]:
# import pandas as pd
# import numpy as np
# from sklearn.metrics.pairwise import cosine_similarity

# # ── 1. Carregar dados ─────────────────────────────────────────────────────────

# df_vendas   = pd.read_csv('../data/raw/vendas_2023_2024.csv')
# df_produtos = pd.read_csv('../data/raw/produtos_raw.csv')
# df_produtos = df_produtos.drop_duplicates(subset='code', keep='first')

# PRODUTO_ALVO = 'GPS Garmin Vortex Maré Drift'
# TARGET_ID    = df_produtos[df_produtos['name'] == PRODUTO_ALVO]['code'].iloc[0]

# # ── 2. Matriz usuário × produto (binária) ─────────────────────────────────────

# df_interacao = df_vendas.groupby(['id_client', 'id_product'])['qtd'].sum().reset_index()
# df_interacao['comprou'] = 1

# matriz = df_interacao.pivot_table(
#     index='id_client',
#     columns='id_product',
#     values='comprou',
#     fill_value=0
# )

# print(f'Matriz: {matriz.shape[0]} clientes × {matriz.shape[1]} produtos')

# # ── 3. Similaridade de Cosseno entre produtos ─────────────────────────────────

# sim_matrix = cosine_similarity(matriz.T)

# sim_df = pd.DataFrame(
#     sim_matrix,
#     index=matriz.columns,
#     columns=matriz.columns
# )

# # ── 4. Ranking dos 5 produtos mais similares ──────────────────────────────────

# ranking = (
#     sim_df[TARGET_ID]
#     .drop(TARGET_ID)
#     .sort_values(ascending=False)
#     .head(5)
#     .reset_index()
# )
# ranking.columns = ['id_produto', 'similaridade']
# ranking = ranking.merge(df_produtos[['code', 'name']], left_on='id_produto', right_on='code').drop(columns='code')
# ranking.index += 1

# # ── 5. Resultado ──────────────────────────────────────────────────────────────

# print(f'\nProduto de referência: {PRODUTO_ALVO} (id={TARGET_ID})')
# print('\nQuem comprou isso, também levou:')
# print('-' * 60)
# for i, row in ranking.iterrows():
#     print(f'  {i}. {row["name"]}')
#     print(f'     Similaridade: {row["similaridade"]:.4f}')

### Questão 8.2
 O produto com maior similaridade é o id_produto 94 — Motor de Popa Volvo Magnum 276HP, com similaridade de 0,8696.

### Questão 8.3
A matriz foi construída agrupando as vendas por id_client e id_product, binarizando o resultado — qualquer quantidade comprada vira 1, zero compras vira 0. Em seguida, um pivot_table organiza os clientes nas linhas e os produtos nas colunas, resultando em uma matriz onde cada célula indica se aquele cliente já comprou aquele produto ao menos uma vez.

A similaridade de cosseno nesse contexto mede o quanto dois produtos foram comprados pelos mesmos clientes. Cada produto é representado por um vetor de 49 posições — uma por cliente — onde 1 significa que o cliente comprou e 0 que não comprou. Dois produtos com similaridade 1.0 foram comprados exatamente pelo mesmo conjunto de clientes; similaridade 0.0 significa que nenhum cliente comprou os dois. O GPS Garmin e o Motor de Popa Volvo terem similaridade 0.87 indica que a grande maioria dos clientes que comprou um também comprou o outro.

A principal limitação é o problema de cold start: o método só consegue recomendar produtos que já foram comprados por pelo menos um cliente em comum com o produto de referência. Um produto novo, sem histórico de compras, terá vetor de zeros e similaridade zero com tudo e nunca aparecerá em nenhum ranking, independentemente de quão relevante ele seja para o contexto.